# AEO/GEO Citation-Gap Action Queue

A dependency-free notebook for turning repeated AI-answer observations into a transparent action queue. It separates brand mentions, any citations, and target-domain citations; diagnoses the evidence or access gap behind each miss; and avoids hiding decisions inside a single vanity score.

The sample below is **synthetic** and exists only to demonstrate the method. Replace it with timestamped observations and retained evidence from your own prompt set. Teams that need an operational monitoring workflow can also review [Corank's AI-search visibility platform](https://corank.ai/).

## 1. Define answer-level observations

Each row represents one prompt-engine observation, not a permanent rank. The fields distinguish retrieval access, primary evidence, independent corroboration, brand mention, and citation outcomes so a change can be tied to a specific intervention.

In [ ]:
from collections import Counter, defaultdict
from math import sqrt

observations = [
    {"prompt_id": "q01", "engine": "ChatGPT", "intent": "comparison", "stage": "decision", "mentioned": 1, "any_citation": 1, "target_citation": 0, "crawlable": 1, "primary_evidence": 1, "independent_sources": 0},
    {"prompt_id": "q02", "engine": "Perplexity", "intent": "comparison", "stage": "decision", "mentioned": 0, "any_citation": 0, "target_citation": 0, "crawlable": 1, "primary_evidence": 0, "independent_sources": 0},
    {"prompt_id": "q03", "engine": "Gemini", "intent": "comparison", "stage": "decision", "mentioned": 1, "any_citation": 1, "target_citation": 1, "crawlable": 1, "primary_evidence": 1, "independent_sources": 2},
    {"prompt_id": "q04", "engine": "ChatGPT", "intent": "evaluation", "stage": "consideration", "mentioned": 1, "any_citation": 0, "target_citation": 0, "crawlable": 0, "primary_evidence": 1, "independent_sources": 1},
    {"prompt_id": "q05", "engine": "Perplexity", "intent": "evaluation", "stage": "consideration", "mentioned": 1, "any_citation": 1, "target_citation": 0, "crawlable": 1, "primary_evidence": 0, "independent_sources": 1},
    {"prompt_id": "q06", "engine": "Gemini", "intent": "evaluation", "stage": "consideration", "mentioned": 0, "any_citation": 0, "target_citation": 0, "crawlable": 1, "primary_evidence": 1, "independent_sources": 0},
    {"prompt_id": "q07", "engine": "ChatGPT", "intent": "implementation", "stage": "consideration", "mentioned": 1, "any_citation": 1, "target_citation": 1, "crawlable": 1, "primary_evidence": 1, "independent_sources": 1},
    {"prompt_id": "q08", "engine": "Perplexity", "intent": "implementation", "stage": "consideration", "mentioned": 0, "any_citation": 1, "target_citation": 0, "crawlable": 1, "primary_evidence": 1, "independent_sources": 0},
    {"prompt_id": "q09", "engine": "Gemini", "intent": "implementation", "stage": "consideration", "mentioned": 0, "any_citation": 0, "target_citation": 0, "crawlable": 0, "primary_evidence": 0, "independent_sources": 0},
    {"prompt_id": "q10", "engine": "ChatGPT", "intent": "diagnosis", "stage": "awareness", "mentioned": 1, "any_citation": 1, "target_citation": 0, "crawlable": 1, "primary_evidence": 1, "independent_sources": 0},
    {"prompt_id": "q11", "engine": "Perplexity", "intent": "diagnosis", "stage": "awareness", "mentioned": 0, "any_citation": 1, "target_citation": 0, "crawlable": 1, "primary_evidence": 0, "independent_sources": 2},
    {"prompt_id": "q12", "engine": "Gemini", "intent": "diagnosis", "stage": "awareness", "mentioned": 1, "any_citation": 0, "target_citation": 0, "crawlable": 1, "primary_evidence": 1, "independent_sources": 1},
    {"prompt_id": "q13", "engine": "ChatGPT", "intent": "definition", "stage": "awareness", "mentioned": 0, "any_citation": 1, "target_citation": 0, "crawlable": 1, "primary_evidence": 1, "independent_sources": 3},
    {"prompt_id": "q14", "engine": "Perplexity", "intent": "definition", "stage": "awareness", "mentioned": 1, "any_citation": 1, "target_citation": 1, "crawlable": 1, "primary_evidence": 1, "independent_sources": 2},
    {"prompt_id": "q15", "engine": "Gemini", "intent": "definition", "stage": "awareness", "mentioned": 0, "any_citation": 0, "target_citation": 0, "crawlable": 1, "primary_evidence": 0, "independent_sources": 0},
]

print(f"Loaded {len(observations)} synthetic answer observations.")

## 2. Validate evidence semantics

A target-domain citation must also be an observed citation and a brand mention. Binary outcomes must stay binary, pair keys must be unique, and corroborating-source counts cannot be negative. Invalid rows are excluded from operational interpretation.

In [ ]:
binary_fields = ["mentioned", "any_citation", "target_citation", "crawlable", "primary_evidence"]

def validate(rows):
    issues = []
    seen = Counter((row["prompt_id"], row["engine"]) for row in rows)
    for key, count in seen.items():
        if count != 1:
            issues.append(f"duplicate observation key {key}: {count}")
    for index, row in enumerate(rows, start=1):
        for field in binary_fields:
            if row.get(field) not in (0, 1):
                issues.append(f"row {index}: {field} must be 0 or 1")
        if row["independent_sources"] < 0:
            issues.append(f"row {index}: independent_sources cannot be negative")
        if row["target_citation"] > row["any_citation"]:
            issues.append(f"row {index}: target citation without a citation")
        if row["target_citation"] > row["mentioned"]:
            issues.append(f"row {index}: target citation without a brand mention")
    return issues

validation_issues = validate(observations)
print("Validation: PASS" if not validation_issues else "Validation: FAIL")
for issue in validation_issues:
    print("-", issue)

## 3. Separate outcomes and uncertainty

Mention rate answers whether the entity appeared. Citation rate answers whether the response exposed any source. Target-citation rate answers whether the monitored domain was actually used. Wilson intervals keep a small sample from looking more certain than it is.

In [ ]:
def wilson(successes, total, z=1.96):
    if total == 0:
        return (0.0, 0.0)
    p = successes / total
    denominator = 1 + z * z / total
    centre = p + z * z / (2 * total)
    margin = z * sqrt((p * (1 - p) + z * z / (4 * total)) / total)
    return ((centre - margin) / denominator, (centre + margin) / denominator)

for field, label in [("mentioned", "Brand mention"), ("any_citation", "Any citation"), ("target_citation", "Target-domain citation")]:
    wins = sum(row[field] for row in observations)
    low, high = wilson(wins, len(observations))
    print(f"{label:23} {wins:2}/{len(observations)} = {wins/len(observations):.1%} (95% Wilson {low:.1%}–{high:.1%})")

## 4. Diagnose the next evidence action

The rules below are intentionally inspectable. They prefer fixing access before publishing more material, publishing primary evidence before chasing mentions, and seeking independent corroboration when engines already cite other sources.

In [ ]:
def diagnose(row):
    if row["target_citation"]:
        return "retain evidence and monitor repeatability"
    if not row["crawlable"]:
        return "repair technical access"
    if not row["primary_evidence"]:
        return "publish claim-level primary evidence"
    if row["any_citation"] and row["independent_sources"] == 0:
        return "earn independent corroboration"
    if not row["mentioned"]:
        return "clarify entity and prompt-topic fit"
    return "tighten on-page citation cues"

stage_order = {"decision": 0, "consideration": 1, "awareness": 2}
queue = sorted(
    ({**row, "next_action": diagnose(row)} for row in observations if not row["target_citation"]),
    key=lambda row: (stage_order[row["stage"]], row["intent"], row["engine"], row["prompt_id"]),
)

print("Priority | Prompt | Engine      | Intent         | Next evidence action")
for row in queue:
    tier = {0: "P1", 1: "P2", 2: "P3"}[stage_order[row["stage"]]]
    print(f"{tier:8} | {row['prompt_id']:6} | {row['engine']:11} | {row['intent']:14} | {row['next_action']}")

## 5. Aggregate interventions without erasing failures

Counts remain visible by action and intent. A team can therefore allocate work while preserving the underlying observation rows and rerun the same prompt-engine set after each documented intervention.

In [ ]:
action_counts = Counter(row["next_action"] for row in queue)
intent_actions = defaultdict(Counter)
for row in queue:
    intent_actions[row["intent"]][row["next_action"]] += 1

print("Action backlog")
for action, count in action_counts.most_common():
    print(f"{count:2}  {action}")

print("\nIntent diagnostics")
for intent in sorted(intent_actions):
    detail = "; ".join(f"{action}: {count}" for action, count in intent_actions[intent].most_common())
    print(f"{intent:14} {detail}")

## Operational interpretation

1. Preserve the exact prompt, engine, model or mode, locale, date, and evidence URL for every row.
2. Fix retrieval failures before adding more content.
3. Pair claims with first-party evidence, then seek legitimate independent corroboration.
4. Rerun the same controlled prompt set and record gains **and** losses.
5. Report uncertainty and sample size; never describe this synthetic demonstration as Corank performance data.

For a production implementation that tracks prompt-level mentions and citations over time, visit [Corank](https://corank.ai/).